# R notebook for regression, clustering, and figure export

This notebook is split into separate cells and does the following:

- prepares the 1997–2023 dataset and saves it to CSV
- runs regression equation (5)
- runs regression equation (6) based on principal components
- prepares the standardized datasets used for the coefficient variation diagrams
- saves all figure outputs as **PDF** files using `cairo_pdf(..., fallback_resolution = 300)`
- saves key text outputs to `.txt` files

All generated files are written into these folders relative to the notebook location:

- `data/`
- `outputs/`


The clustering section has been revised using fixed seeds, repeated independent k-means runs, paired original-versus-sparsified comparisons, and 95% confidence intervals.


In [1]:
# Setup ----------------------------------------------------------------------

dir.create("data", showWarnings = FALSE)
dir.create("outputs_new", showWarnings = FALSE)

install_if_missing <- function(pkgs) {
  to_install <- pkgs[!pkgs %in% rownames(installed.packages())]
  if (length(to_install) > 0) {
    install.packages(to_install, repos = "https://cloud.r-project.org")
  }
}

install_if_missing(c("lars", "glmnet"))

library(lars)
library(glmnet)

save_plot_pdf <- function(filename, plot_expr, width = 7, height = 6) {
  grDevices::cairo_pdf(
    filename = file.path("outputs", filename),
    width = width,
    height = height,
    fallback_resolution = 300
  )
  on.exit(dev.off(), add = TRUE)
  eval.parent(substitute(plot_expr))
}

standardize_for_path <- function(v) {
  v <- v - mean(v)
  k <- length(v) / sqrt(drop(crossprod(v)))
  v * k
}

Loaded lars 1.3


Warning message:
"package 'glmnet' was built under R version 4.4.3"
Loading required package: Matrix

Loaded glmnet 4.1-10



## 1. Regression equation (5): prepare dataset and save CSV

In [2]:
# Regression equation (5) data ------------------------------------------------

year <- 1997:2023

# 1997-2023 price of commercial housing
y <- c(1997,2063,2053,2112,2170,2250,2359,2714,3168,3367,3864,3800,4681,5045,5384,5839,6328,6427,6932,7699,8160,9045,9673,10248,10546,10210,10438) / 1997

# 1997-2023 Chinese gross domestic product
x1 <- c(79715.0,85195.5,90564.4,100280.1,110863.1,121717.4,137422.0,161840.2,187318.9,219438.5,270092.3,319244.6,348517.7,412119.3,487940.2,538580.0,592963.2,643563.1,688858.2,746395.1,832035.9,919281.1,986515.2,1013567.0,1149237.0,1204724.0,1260582.1) / 79715.0

# 1997-2023 Chinese fiscal revenue
x2 <- c(8651.14,9875.95,11444.08,13395.23,16386.04,18903.64,21715.25,26396.47,31649.29,38760.20,51321.78,61330.35,68518.30,83101.51,103874.43,117253.52,129209.64,140370.03,152269.23,159269.97,172592.77,183359.84,190390.08,182913.88,202554.64,203649.29,216795.43) / 8651.14

# 1997-2023 Chinese broad money supply
x3 <- c(90995.3,104498.5,119897.9,134610.3,158301.9,185007.0,221222.8,254107.0,298755.7,345577.9,403442.2,475166.6,610224.5,725851.8,851590.9,974148.8,1106525.0,1228374.8,1392278.1,1550066.7,1690235.3,1826744.2,1986488.8,2186795.9,2382899.6,2664320.8,2922713.3) / 90995.3

# 1997-2023 Chinese consumer price index
x4 <- c(100,99.2,98.6,100.4,100.7,99.2,101.2,103.9,101.8,101.5,104.8,105.9,99.3,103.3,105.4,102.6,102.6,102.0,101.4,102.0,101.6,102.1,102.9,102.5,100.9,102.0,100.2) / 100
for (i in 2:27) {
  x4[i] <- x4[i - 1] * x4[i]
}

# 1997-2023 fixed assets investment of the whole society in China
x5 <- c(24941,28406,29855,32918,37214,43500,53841,66235,80994,97583,118323,144587,181760,218834,205036,241746,282486,320331,347827,372021,394926,418215,439541,451155,473003,495966,509708) / 24941

# 1997-2023 the cost of commercial housing in China
x6 <- c(1175,1218,1152,1139,1128,1184,1273,1402,1451,1564,1657,1795,2021,2228,2373,2498,2643,2816,3054,3039,3105,3210,3549,3781,3891,4093,4150) / 1175

# 1997-2023 per capita disposable income of urban residents in China
x7 <- c(5160.3,5425.1,5854.0,6280.0,6859.6,7702.8,8472.2,9421.6,10493.0,11759.5,13785.8,15780.8,17174.7,19109.4,21809.8,24564.7,26955.1,28843.9,31194.8,33616.2,36396.2,39250.8,42358.8,43833.8,47412,49283,51821) / 5160.3

X <- data.frame(year, x1, x2, x3, x4, x5, x6, x7, y)

write.csv(X, file = file.path("data", "regression_data_1997_2023.csv"), row.names = FALSE)

X

year,x1,x2,x3,x4,x5,x6,x7,y
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1997,1.000000,1.000000,1.000000,1.0000000,1.000000,1.0000000,1.000000,1.000000
1998,1.068751,1.141578,1.148394,0.9920000,1.138928,1.0365957,1.051315,1.033050
1999,1.136102,1.322841,1.317627,0.9781120,1.197025,0.9804255,1.134430,1.028042
2000,1.257983,1.548377,1.479310,0.9820244,1.319835,0.9693617,1.216984,1.057586
2001,1.390743,1.894090,1.739671,0.9888986,1.492081,0.9600000,1.329303,1.086630
2002,1.526907,2.185104,2.033149,0.9809874,1.744116,1.0076596,1.492704,1.126690
2003,1.723916,2.510103,2.431145,0.9927593,2.158735,1.0834043,1.641804,1.181272
2004,2.030235,3.051213,2.792529,1.0314769,2.655667,1.1931915,1.825785,1.359039
2005,2.349858,3.658395,3.283199,1.0500435,3.247424,1.2348936,2.033409,1.586380


## 2. Fit regression equation (5) and save summary

In [3]:
# Regression equation (5) -----------------------------------------------------

hp <- lm(y ~ x1 + x2 + x3 + x4 + x5 + x6 + x7, data = X)

summary_hp <- summary(hp)
summary_hp

capture.output(summary_hp, file = file.path("outputs", "regression_equation_5_summary.txt"))


Call:
lm(formula = y ~ x1 + x2 + x3 + x4 + x5 + x6 + x7, data = X)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.26512 -0.05632  0.01297  0.05291  0.23561 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)   
(Intercept)  0.64371    1.53949   0.418  0.68054   
x1           0.11011    0.18589   0.592  0.56062   
x2          -0.08653    0.04368  -1.981  0.06223 . 
x3          -0.13567    0.04616  -2.939  0.00842 **
x4          -0.78316    1.94359  -0.403  0.69149   
x5           0.07412    0.06338   1.169  0.25668   
x6           0.23589    0.49223   0.479  0.63725   
x7           0.83106    0.33954   2.448  0.02427 * 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.1206 on 19 degrees of freedom
Multiple R-squared:  0.9955,	Adjusted R-squared:  0.9938 
F-statistic: 596.3 on 7 and 19 DF,  p-value: < 2.2e-16


## 3. Fit regression equation (6) via principal components and save summary

In [4]:
# Regression equation (6) -----------------------------------------------------

hp.pr <- princomp(~ x1 + x2 + x3 + x4 + x5 + x6 + x7, data = X, cor = TRUE)
summary_hp_pr <- summary(hp.pr, loadings = TRUE)
summary_hp_pr

pre <- predict(hp.pr)
X$z1 <- pre[, 1]

lm.sol <- lm(y ~ z1, data = X)
summary_lm_sol <- summary(lm.sol)
summary_lm_sol

beta <- coef(lm.sol)
A <- loadings(hp.pr)
x.bar <- hp.pr$center
x.sd <- hp.pr$scale
coef_pc <- beta[2] * A[, 1] / x.sd
beta0 <- beta[1] - sum(x.bar * coef_pc)

pc_regression_coefficients <- c(beta0 = beta0, coef_pc)
pc_regression_coefficients

capture.output(summary_hp_pr, file = file.path("outputs", "regression_equation_6_pca_summary.txt"))
capture.output(summary_lm_sol, file = file.path("outputs", "regression_equation_6_lm_summary.txt"))
write.csv(
  data.frame(term = names(pc_regression_coefficients), value = as.numeric(pc_regression_coefficients)),
  file = file.path("outputs", "regression_equation_6_coefficients.csv"),
  row.names = FALSE
)

Importance of components:
                          Comp.1      Comp.2      Comp.3       Comp.4
Standard deviation     2.6349591 0.206567137 0.085060511 0.0620737031
Proportion of Variance 0.9918585 0.006095712 0.001033613 0.0005504492
Cumulative Proportion  0.9918585 0.997954172 0.998987785 0.9995382345
                             Comp.5       Comp.6       Comp.7
Standard deviation     0.0428412881 0.0326148374 0.0182552614
Proportion of Variance 0.0002621966 0.0001519611 0.0000476078
Cumulative Proportion  0.9998004311 0.9999523922 1.0000000000

Loadings:
   Comp.1 Comp.2 Comp.3 Comp.4 Comp.5 Comp.6 Comp.7
x1  0.379  0.309         0.339  0.293  0.370  0.647
x2  0.377 -0.435  0.512  0.251 -0.578              
x3  0.376  0.674               -0.328 -0.536       
x4  0.378 -0.414 -0.464  0.409  0.245 -0.493       
x5  0.378 -0.249  0.315 -0.697  0.381 -0.193  0.166
x6  0.379        -0.629 -0.382 -0.374  0.417       
x7  0.379  0.162  0.144  0.138  0.357  0.336 -0.741


Call:
lm(formula = y ~ z1, data = X)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.29860 -0.05156 -0.01894  0.10229  0.34340 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  2.75547    0.02894   95.21   <2e-16 ***
z1           0.56788    0.01098   51.70   <2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.1504 on 25 degrees of freedom
Multiple R-squared:  0.9907,	Adjusted R-squared:  0.9904 
F-statistic:  2673 on 1 and 25 DF,  p-value: < 2.2e-16


beta0.(Intercept)                x1                x2                x3 
      -0.38999198        0.04498709        0.02562115        0.02259855 
               x4                x5                x6                x7 
       0.97450980        0.03190381        0.25029377        0.07354322

## 4. Prepare standardized data for coefficient variation diagrams (Figures 1–3)

In [5]:
# Standardized data for coefficient variation diagrams 1, 2, 3 ---------------

y_std <- c(1997,2063,2053,2112,2170,2250,2359,2714,3168,3367,3864,3800,4681,5045,5384,5839,6328,6427,6932,7699,8160,9045,9673,10248,10546,10210,10438) / 1997
y_std <- y_std - mean(y_std)

x1_std <- c(79715.0,85195.5,90564.4,100280.1,110863.1,121717.4,137422.0,161840.2,187318.9,219438.5,270092.3,319244.6,348517.7,412119.3,487940.2,538580.0,592963.2,643563.1,688858.2,746395.1,832035.9,919281.1,986515.2,1013567.0,1149237.0,1204724.0,1260582.1) / 79715.0
x1_std <- standardize_for_path(x1_std)

x2_std <- c(8651.14,9875.95,11444.08,13395.23,16386.04,18903.64,21715.25,26396.47,31649.29,38760.20,51321.78,61330.35,68518.30,83101.51,103874.43,117253.52,129209.64,140370.03,152269.23,159269.97,172592.77,183359.84,190390.08,182913.88,202554.64,203649.29,216795.43) / 8651.14
x2_std <- standardize_for_path(x2_std)

x3_std <- c(90995.3,104498.5,119897.9,134610.3,158301.9,185007.0,221222.8,254107.0,298755.7,345577.9,403442.2,475166.6,610224.5,725851.8,851590.9,974148.8,1106525.0,1228374.8,1392278.1,1550066.7,1690235.3,1826744.2,1986488.8,2186795.9,2382899.6,2664320.8,2922713.3) / 90995.3
x3_std <- standardize_for_path(x3_std)

x4_std <- c(100,99.2,98.6,100.4,100.7,99.2,101.2,103.9,101.8,101.5,104.8,105.9,99.3,103.3,105.4,102.6,102.6,102.0,101.4,102.0,101.6,102.1,102.9,102.5,100.9,102.0,100.2) / 100
for (i in 2:27) {
  x4_std[i] <- x4_std[i - 1] * x4_std[i]
}
x4_std <- standardize_for_path(x4_std)

x5_std <- c(24941,28406,29855,32918,37214,43500,53841,66235,80994,97583,118323,144587,181760,218834,205036,241746,282486,320331,347827,372021,394926,418215,439541,451155,473003,495966,509708) / 24941
x5_std <- standardize_for_path(x5_std)

x6_std <- c(1175,1218,1152,1139,1128,1184,1273,1402,1451,1564,1657,1795,2021,2228,2373,2498,2643,2816,3054,3039,3105,3210,3549,3781,3891,4093,4150) / 1175
x6_std <- standardize_for_path(x6_std)

x7_std <- c(5160.3,5425.1,5854.0,6280.0,6859.6,7702.8,8472.2,9421.6,10493.0,11759.5,13785.8,15780.8,17174.7,19109.4,21809.8,24564.7,26955.1,28843.9,31194.8,33616.2,36396.2,39250.8,42358.8,43833.8,47412,49283,51821) / 5160.3
x7_std <- standardize_for_path(x7_std)

X_std <- cbind(x1_std, x2_std, x3_std, x4_std, x5_std, x6_std, x7_std)

standardized_data <- data.frame(
  year = 1997:2023,
  x1 = x1_std, x2 = x2_std, x3 = x3_std, x4 = x4_std,
  x5 = x5_std, x6 = x6_std, x7 = x7_std, y = y_std
)

write.csv(
  standardized_data,
  file = file.path("data", "standardized_data_1997_2023.csv"),
  row.names = FALSE
)

head(standardized_data)

,year,x1,x2,x3,x4,x5,x6,x7,y
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1997,-5.838348,-6.334849,-5.473377,-5.796556,-6.228377,-5.882467,-6.130846,-1.755467
2,1998,-5.763590,-6.246915,-5.391707,-5.985456,-6.121216,-5.661189,-6.039772,-1.722417
3,1999,-5.690355,-6.134332,-5.298569,-6.313385,-6.076403,-6.000826,-5.892258,-1.727424
4,2000,-5.557826,-5.994250,-5.209585,-6.221003,-5.981674,-6.067724,-5.745742,-1.697880
5,2001,-5.413467,-5.779527,-5.066294,-6.058687,-5.848812,-6.124330,-5.546397,-1.668837
6,2002,-5.265407,-5.598777,-4.904777,-6.245490,-5.654406,-5.836153,-5.256391,-1.628776


## 5. Generate and save Figures 1–3 as PDF

In [6]:
# Figures 1, 2, 3 ------------------------------------------------------------

c_lasso_1 <- lars(X_std, y_std, type = "lasso")
a_lar_1 <- lars(X_std, y_std, type = "lar")
b_glmnet_1 <- glmnet(X_std, y_std, alpha = 1, family = "gaussian")

save_plot_pdf("figure1_lasso_path_1997_2023.pdf", plot(c_lasso_1))
save_plot_pdf("figure2_lar_path_1997_2023.pdf", plot(a_lar_1))
save_plot_pdf("figure3_glmnet_path_1997_2023.pdf", plot(b_glmnet_1))

lar_beta_1 <- lars(X_std, y_std, type = "lar")$beta
lasso_beta_1 <- lars(X_std, y_std, type = "lasso")$beta
glmnet_beta_1 <- as.matrix(glmnet(X_std, y_std, alpha = 1, family = "gaussian")$beta)

lar_beta_1
lasso_beta_1
glmnet_beta_1

write.csv(lar_beta_1, file = file.path("outputs", "figure1_3_lar_beta.csv"), row.names = FALSE)
write.csv(lasso_beta_1, file = file.path("outputs", "figure1_3_lasso_beta.csv"), row.names = FALSE)
write.csv(glmnet_beta_1, file = file.path("outputs", "figure1_3_glmnet_beta.csv"), row.names = TRUE)

,x1_std,x2_std,x3_std,x4_std,x5_std,x6_std,x7_std
0,0.000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000
1,0.000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.09863443
2,0.000000,0.00000000,0.00000000,0.000000000,0.05725972,0.00000000,0.15589415
3,0.000000,0.00000000,0.00000000,0.032754300,0.06673289,0.00000000,0.18879449
4,0.000000,0.00000000,-0.05080159,0.009595910,0.04943748,0.00000000,0.27954915
5,0.000000,0.00000000,-0.07630591,-0.008246843,0.03994435,0.01455839,0.31756072
6,0.000000,-0.06060263,-0.14021239,-0.015086558,0.05682650,0.02396027,0.42182469
7,0.101259,-0.13931289,-0.24651985,-0.033167085,0.09609610,0.03901240,0.46825608


,x1_std,x2_std,x3_std,x4_std,x5_std,x6_std,x7_std
0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000
1,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000000,0.09863443
2,0.00000000,0.00000000,0.00000000,0.00000000,0.05725972,0.000000000,0.15589415
3,0.00000000,0.00000000,0.00000000,0.03275430,0.06673289,0.000000000,0.18879449
4,0.00000000,0.00000000,-0.05080159,0.00959591,0.04943748,0.000000000,0.27954915
5,0.00000000,0.00000000,-0.06451792,0.00000000,0.04433203,0.007829564,0.29999194
6,0.00000000,0.00000000,-0.06928378,0.00000000,0.04064973,0.008314629,0.30792361
7,0.00000000,-0.07152128,-0.13783090,0.00000000,0.06126393,0.013299038,0.42153979
8,0.05933819,-0.12531396,-0.19845498,0.00000000,0.08739248,0.014632282,0.44854869
9,0.10125900,-0.13931289,-0.24651985,-0.03316708,0.09609610,0.039012401,0.46825608


,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,⋯,s45,s46,s47,s48,s49,s50,s51,s52,s53,s54
x1_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x2_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x3_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x4_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.02270462,0.02285529,0.02300565,0.02320507,0.02335521,0.02350544,0.02365572,0.02375635,0.02385708,0.02395792
x5_std,0,0.00000000,0.00000000,0.00000000,0.003777238,0.01262400,0.02072126,0.02804779,0.03471738,0.04080972,⋯,0.07728137,0.07735347,0.07739074,0.07730641,0.07728727,0.07724358,0.07717816,0.07718226,0.07716711,0.07713477
x6_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x7_std,0,0.02559004,0.04890673,0.07015203,0.085752905,0.09459165,0.10260895,0.10996514,0.11667386,0.12277143,⋯,0.18430913,0.18447705,0.18464537,0.18485453,0.18501901,0.18518169,0.18534210,0.18546087,0.18557891,0.18569593


## 6. Prepare data for Figures 4–6 and save CSV

In [7]:
# Figures 4, 5, 6 data --------------------------------------------------------

x1_f46 <- c(10,20,30,0,0,0,0,0,0,0,0,0,0,0,0,0)
x1_f46 <- standardize_for_path(x1_f46)

x2_f46 <- c(0,0,0,40,50,60,70,80,9,10,0,0,0,0,0,0)
x2_f46 <- standardize_for_path(x2_f46)

x3_f46 <- c(0,0,0,0,0,0,0,0,0,0,11,12,13,0,0,0)
x3_f46 <- standardize_for_path(x3_f46)

x4_f46 <- c(0,0,0,0,0,0,0,0,0,0,0,0,0,14,15,16)
x4_f46 <- standardize_for_path(x4_f46)

X_f46 <- cbind(x1_f46, x2_f46, x3_f46, x4_f46)
y_f46 <- c(-1,10,-5,-10,1,8,3,4,2,1,9,7,5,3,2,-1)
y_f46 <- y_f46 - mean(y_f46)

figure46_data <- data.frame(
  obs = 1:16,
  x1 = x1_f46, x2 = x2_f46, x3 = x3_f46, x4 = x4_f46, y = y_f46
)

write.csv(
  figure46_data,
  file = file.path("data", "figure4_6_data.csv"),
  row.names = FALSE
)

figure46_data

obs,x1,x2,x3,x4,y
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2.91730,-2.817285,-1.916087,-1.918044,-3.375
2,7.58498,-2.817285,-1.916087,-1.918044,7.625
3,12.25266,-2.817285,-1.916087,-1.918044,-7.375
4,-1.75038,2.834948,-1.916087,-1.918044,-12.375
5,-1.75038,4.248007,-1.916087,-1.918044,-1.375
6,-1.75038,5.661065,-1.916087,-1.918044,5.625
7,-1.75038,7.074123,-1.916087,-1.918044,0.625
8,-1.75038,8.487182,-1.916087,-1.918044,1.625
9,-1.75038,-1.545533,-1.916087,-1.918044,-0.375


## 7. Generate and save Figures 4–6 as PDF

In [8]:
# Figures 4, 5, 6 ------------------------------------------------------------

c_lasso_46 <- lars(X_f46, y_f46, type = "lasso")
a_lar_46 <- lars(X_f46, y_f46, type = "lar")
b_glmnet_46 <- glmnet(X_f46, y_f46, alpha = 1, family = "gaussian")

save_plot_pdf("figure4_lasso_path_block.pdf", plot(c_lasso_46))
save_plot_pdf("figure5_lar_path_block.pdf", plot(a_lar_46))
save_plot_pdf("figure6_glmnet_path_block.pdf", plot(b_glmnet_46))

lar_beta_46 <- lars(X_f46, y_f46, type = "lar")$beta
lasso_beta_46 <- lars(X_f46, y_f46, type = "lasso")$beta
glmnet_beta_46 <- as.matrix(glmnet(X_f46, y_f46, alpha = 1, family = "gaussian")$beta)

lar_beta_46
lasso_beta_46
glmnet_beta_46

write.csv(lar_beta_46, file = file.path("outputs", "figure4_6_lar_beta.csv"), row.names = FALSE)
write.csv(lasso_beta_46, file = file.path("outputs", "figure4_6_lasso_beta.csv"), row.names = FALSE)
write.csv(glmnet_beta_46, file = file.path("outputs", "figure4_6_glmnet_beta.csv"), row.names = TRUE)

,x1_f46,x2_f46,x3_f46,x4_f46
0,0.000000000,0.00000000,0.0000000,0.00000000
1,0.000000000,0.00000000,0.4329299,0.00000000
2,0.000000000,0.07937655,0.5123064,0.00000000
3,-0.007303067,0.11630382,0.5497721,0.00000000
4,0.037686411,0.22368302,0.6460540,0.09631987


,x1_f46,x2_f46,x3_f46,x4_f46
0,0.000000000,0.00000000,0.0000000,0.00000000
1,0.000000000,0.00000000,0.4329299,0.00000000
2,0.000000000,0.07937655,0.5123064,0.00000000
3,-0.007303067,0.11630382,0.5497721,0.00000000
4,0.000000000,0.13373451,0.5654013,0.01563544
5,0.000000000,0.17828899,0.6066248,0.05687138
6,0.037686411,0.22368302,0.6460540,0.09631987


,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,⋯,s62,s63,s64,s65,s66,s67,s68,s69,s70,s71
x1_f46,0,0.0000000,0.00000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000,⋯,0.02827311,0.02904884,0.02979361,0.03048638,0.03092923,0.03163398,0.03203542,0.03246456,0.03288872,0.03329325
x2_f46,0,0.0000000,0.00000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000,⋯,0.21236521,0.21330012,0.21419622,0.21502920,0.21557767,0.21641122,0.21690640,0.21742451,0.21793464,0.21842028
x3_f46,0,0.0480221,0.09177804,0.1316468,0.1679738,0.2010735,0.2312328,0.2587128,0.2837516,0.306566,⋯,0.63630563,0.63712015,0.63789462,0.63861232,0.63910888,0.63981211,0.64025330,0.64070703,0.64114843,0.64156588
x4_f46,0,0.0000000,0.00000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000,⋯,0.08659350,0.08740921,0.08818279,0.08889895,0.08940471,0.09009909,0.09054618,0.09100132,0.09144226,0.09185835


## 8. Prepare data for Figures 7–9 and save CSV

In [9]:
# Figures 7, 8, 9 data --------------------------------------------------------

x1_f79  <- standardize_for_path(c(10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0))
x2_f79  <- standardize_for_path(c(0,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0))
x3_f79  <- standardize_for_path(c(0,0,30,0,0,0,0,0,0,0,0,0,0,0,0,0))
x4_f79  <- standardize_for_path(c(0,0,0,40,0,0,0,0,0,0,0,0,0,0,0,0))
x5_f79  <- standardize_for_path(c(0,0,0,0,50,0,0,0,0,0,0,0,0,0,0,0))
x6_f79  <- standardize_for_path(c(0,0,0,0,0,60,0,0,0,0,0,0,0,0,0,0))
x7_f79  <- standardize_for_path(c(0,0,0,0,0,0,70,0,0,0,0,0,0,0,0,0))
x8_f79  <- standardize_for_path(c(0,0,0,0,0,0,0,80,0,0,0,0,0,0,0,0))
x9_f79  <- standardize_for_path(c(0,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0))
x10_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,10,0,0,0,0,0,0))
x11_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0))
x12_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,12,0,0,0,0))
x13_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,13,0,0,0))
x14_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,0,14,0,0))
x15_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,0,0,15,0))
x16_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16))

X_f79 <- cbind(
  x1_f79, x2_f79, x3_f79, x4_f79, x5_f79, x6_f79, x7_f79, x8_f79,
  x9_f79, x10_f79, x11_f79, x12_f79, x13_f79, x14_f79, x15_f79, x16_f79
)

y_f79 <- c(-1,10,-5,-10,1,8,3,4,2,1,9,7,5,3,2,-1)
y_f79 <- y_f79 - mean(y_f79)

figure79_data <- data.frame(
  obs = 1:16,
  x1 = x1_f79, x2 = x2_f79, x3 = x3_f79, x4 = x4_f79,
  x5 = x5_f79, x6 = x6_f79, x7 = x7_f79, x8 = x8_f79,
  x9 = x9_f79, x10 = x10_f79, x11 = x11_f79, x12 = x12_f79,
  x13 = x13_f79, x14 = x14_f79, x15 = x15_f79, x16 = x16_f79,
  y = y_f79
)

write.csv(
  figure79_data,
  file = file.path("data", "figure7_9_data.csv"),
  row.names = FALSE
)

figure79_data

obs,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14,x15,x16,y
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-3.375
2,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,7.625
3,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-7.375
4,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-12.375
5,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.375
6,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,5.625
7,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,0.625
8,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,1.625
9,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-0.375


## 9. Generate and save Figures 7–9 as PDF

In [10]:
# Figures 7, 8, 9 ------------------------------------------------------------

c_lasso_79 <- lars(X_f79, y_f79, type = "lasso")
a_lar_79 <- lars(X_f79, y_f79, type = "lar")
b_glmnet_79 <- glmnet(X_f79, y_f79, alpha = 1, family = "gaussian")

save_plot_pdf("figure7_lasso_path_singletons.pdf", plot(c_lasso_79))
save_plot_pdf("figure8_lar_path_singletons.pdf", plot(a_lar_79))
save_plot_pdf("figure9_glmnet_path_singletons.pdf", plot(b_glmnet_79))

lar_beta_79 <- lars(X_f79, y_f79, type = "lar")$beta
lasso_beta_79 <- lars(X_f79, y_f79, type = "lasso")$beta
glmnet_beta_79 <- as.matrix(glmnet(X_f79, y_f79, alpha = 1, family = "gaussian")$beta)

lar_beta_79
lasso_beta_79
glmnet_beta_79

write.csv(lar_beta_79, file = file.path("outputs", "figure7_9_lar_beta.csv"), row.names = FALSE)
write.csv(lasso_beta_79, file = file.path("outputs", "figure7_9_lasso_beta.csv"), row.names = FALSE)
write.csv(glmnet_beta_79, file = file.path("outputs", "figure7_9_glmnet_beta.csv"), row.names = TRUE)

,x1_f79,x2_f79,x3_f79,x4_f79,x5_f79,x6_f79,x7_f79,x8_f79,x9_f79,x10_f79,x11_f79,x12_f79,x13_f79,x14_f79,x15_f79,x16_f79
0,0.00000000,0.00000000,0.00000000,0.0000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
1,0.00000000,0.00000000,0.00000000,-0.3025768,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
2,0.00000000,0.00000000,-0.03025768,-0.3328345,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
3,0.00000000,0.06051536,-0.10085894,-0.4034358,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
4,0.00000000,0.12103073,-0.16137431,-0.4639511,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.06051536,0.00000000,0.00000000,0.00000000,0,0.00000000
5,0.00000000,0.18154609,-0.21180378,-0.5143806,0.00000000,0.06051536,0.00000000,0.00000000,0.000000e+00,0.00000000,0.12103073,0.00000000,0.00000000,0.00000000,0,0.00000000
6,0.00000000,0.22693262,-0.24206146,-0.5446383,0.00000000,0.10590189,0.00000000,0.00000000,0.000000e+00,0.00000000,0.16641725,0.04538652,0.00000000,0.00000000,0,0.00000000
7,-0.07564421,0.30257682,-0.31770567,-0.6202825,0.00000000,0.18154609,0.00000000,0.00000000,0.000000e+00,0.00000000,0.24206146,0.12103073,0.00000000,0.00000000,0,-0.07564421
8,-0.12103073,0.36309219,-0.36309219,-0.6656690,0.00000000,0.24206146,0.00000000,0.00000000,0.000000e+00,0.00000000,0.30257682,0.18154609,0.06051536,0.00000000,0,-0.12103073
9,-0.18154609,0.42360755,-0.42360755,-0.7261844,-0.06051536,0.30257682,0.00000000,0.06051536,0.000000e+00,-0.06051536,0.36309219,0.24206146,0.12103073,0.00000000,0,-0.18154609


,x1_f79,x2_f79,x3_f79,x4_f79,x5_f79,x6_f79,x7_f79,x8_f79,x9_f79,x10_f79,x11_f79,x12_f79,x13_f79,x14_f79,x15_f79,x16_f79
0,0.00000000,0.00000000,0.00000000,0.0000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
1,0.00000000,0.00000000,0.00000000,-0.3025768,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
2,0.00000000,0.00000000,-0.03025768,-0.3328345,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
3,0.00000000,0.06051536,-0.10085894,-0.4034358,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
4,0.00000000,0.12103073,-0.16137431,-0.4639511,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.06051536,0.00000000,0.00000000,0.00000000,0,0.00000000
5,0.00000000,0.18154609,-0.21180378,-0.5143806,0.00000000,0.06051536,0.00000000,0.00000000,0.000000e+00,0.00000000,0.12103073,0.00000000,0.00000000,0.00000000,0,0.00000000
6,0.00000000,0.22693262,-0.24206146,-0.5446383,0.00000000,0.10590189,0.00000000,0.00000000,0.000000e+00,0.00000000,0.16641725,0.04538652,0.00000000,0.00000000,0,0.00000000
7,-0.07564421,0.30257682,-0.31770567,-0.6202825,0.00000000,0.18154609,0.00000000,0.00000000,0.000000e+00,0.00000000,0.24206146,0.12103073,0.00000000,0.00000000,0,-0.07564421
8,-0.12103073,0.36309219,-0.36309219,-0.6656690,0.00000000,0.24206146,0.00000000,0.00000000,0.000000e+00,0.00000000,0.30257682,0.18154609,0.06051536,0.00000000,0,-0.12103073
9,-0.18154609,0.42360755,-0.42360755,-0.7261844,-0.06051536,0.30257682,0.00000000,0.06051536,0.000000e+00,-0.06051536,0.36309219,0.24206146,0.12103073,0.00000000,0,-0.18154609


,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,⋯,s38,s39,s40,s41,s42,s43,s44,s45,s46,s47
x1_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.189523031,-0.191365207,-0.19304373,-0.19457313,-0.19596667,-0.19723641,-0.19839335,-0.19944751,-0.20040802,-0.20128321
x2_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.009416637,0.04455803,0.07791482,0.10952073,⋯,0.432486580,0.434522985,0.43637848,0.43806914,0.43960961,0.44101322,0.44229215,0.44345745,0.44451924,0.44548669
x3_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,-0.041243202,-0.08224161,-0.11826045,-0.14986711,⋯,-0.431583586,-0.433425842,-0.43510444,-0.43663391,-0.43802751,-0.43929730,-0.44045429,-0.44150850,-0.44246905,-0.44334427
x4_f79,0,-0.07096344,-0.1356227,-0.1945378,-0.248219,-0.2971314,-0.343820247,-0.38481860,-0.42083652,-0.45244290,⋯,-0.734158973,-0.736001356,-0.73768007,-0.73920965,-0.74060334,-0.74187323,-0.74303030,-0.74408458,-0.74504520,-0.74592048
x5_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.068490109,-0.070332479,-0.07201118,-0.07354075,-0.07493443,-0.07620431,-0.07736137,-0.07841564,-0.07937625,-0.08025153
x6_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,0.311456567,0.313492909,0.31534835,0.31703895,0.31857937,0.31998294,0.32126183,0.32242710,0.32348885,0.32445627
x7_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,0.008879842,0.010916175,0.01277161,0.01446221,0.01600262,0.01740618,0.01868506,0.01985032,0.02091207,0.02187949
x8_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,0.069395285,0.071431611,0.07328704,0.07497763,0.07651804,0.07792160,0.07920047,0.08036573,0.08142747,0.08239489
x9_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.007975203,-0.009817532,-0.01149619,-0.01302573,-0.01441938,-0.01568923,-0.01684627,-0.01790051,-0.01886111,-0.01973636
x10_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.068490790,-0.070333099,-0.07201174,-0.07354126,-0.07493490,-0.07620473,-0.07736176,-0.07841599,-0.07937658,-0.08025182


## 10. UCI clustering validation: reproducible repeated-run protocol

This section implements the protocol requested in Reviewer 2, Comment 5.

For every UCI dataset, the number of clusters is fixed to the number of reference classes. The original and sparsified representations are evaluated using:

- **30 independent repetitions**;
- `stats::kmeans` with **30 random starts per repetition**;
- the **Hartigan--Wong** algorithm;
- `iter.max = 100`;
- a fixed master seed and a fully recorded seed for every repetition;
- the **same repetition seeds** for the original and sparsified representations;
- the same observations for silhouette calculation in each paired repetition;
- run-level metrics, mean, standard deviation, and 95% confidence intervals;
- paired differences, defined as sparsified minus original, with 95% confidence intervals.

The four main validation metrics are adjusted Rand index (ARI), clustering accuracy, purity, and average silhouette width. Additional internal metrics are retained for diagnostic purposes.


In [11]:
# Reproducible repeated-run clustering setup ----------------------------------

options(timeout = 600)

install_if_needed <- function(pkgs, repos = "https://cloud.r-project.org") {
  installed <- rownames(installed.packages())
  to_install <- pkgs[!pkgs %in% installed]
  if (length(to_install) > 0) {
    install.packages(to_install, repos = repos)
  }
}

install_if_needed(c("readxl", "cluster", "mclust", "clue"))

library(readxl)
library(cluster)
library(mclust)
library(clue)

# ------------------------------ Protocol constants ---------------------------

SPARSE_ORTHO_BINS <- 4L

CLUSTER_REPETITIONS <- 30L
KMEANS_NSTART <- 30L
KMEANS_ITER_MAX <- 100L
KMEANS_ALGORITHM <- "Hartigan-Wong"

CLUSTER_MASTER_SEED <- 20260716L
SILHOUETTE_SAMPLE_N <- 1000L
CONFIDENCE_LEVEL <- 0.95

clustering_protocol <- data.frame(
  item = c(
    "Independent repetitions",
    "Random starts per repetition",
    "k-means algorithm",
    "Maximum iterations",
    "Master seed",
    "Silhouette sample limit",
    "Confidence level",
    "Sparsification intervals"
  ),
  value = c(
    CLUSTER_REPETITIONS,
    KMEANS_NSTART,
    KMEANS_ALGORITHM,
    KMEANS_ITER_MAX,
    CLUSTER_MASTER_SEED,
    SILHOUETTE_SAMPLE_N,
    CONFIDENCE_LEVEL,
    SPARSE_ORTHO_BINS
  ),
  stringsAsFactors = FALSE
)

write.csv(
  clustering_protocol,
  "outputs/clustering_repeated_run_protocol.csv",
  row.names = FALSE
)

# ------------------------------ File utilities -------------------------------

sanitize_names <- function(x) {
  x <- tolower(trimws(x))
  x <- gsub("[^a-z0-9]+", "_", x)
  x <- gsub("^_+|_+$", "", x)
  x
}

find_data_file <- function(stem, search_dirs = c("data", "."), exts = c("csv", "xlsx", "xls")) {
  for (d in search_dirs) {
    if (!dir.exists(d) && d != ".") next

    for (ext in exts) {
      candidate <- file.path(d, paste0(stem, ".", ext))
      if (file.exists(candidate)) return(candidate)
    }

    pattern <- paste0("^", stem, "\\.(", paste(exts, collapse = "|"), ")$")
    matches <- list.files(
      d,
      pattern = pattern,
      ignore.case = TRUE,
      full.names = TRUE
    )
    if (length(matches) > 0) return(matches[1])
  }
  NA_character_
}

find_data_file_multi <- function(stems, search_dirs = c("data", "."), exts = c("csv", "xlsx", "xls")) {
  for (stem in stems) {
    path <- find_data_file(stem, search_dirs = search_dirs, exts = exts)
    if (!is.na(path)) return(path)
  }
  NA_character_
}

read_tabular_file <- function(path) {
  if (is.na(path)) stop("File not found.")

  ext <- tolower(tools::file_ext(path))
  if (ext %in% c("xlsx", "xls")) {
    as.data.frame(readxl::read_excel(path))
  } else {
    read.csv(path, check.names = FALSE, stringsAsFactors = FALSE)
  }
}

# ------------------------------ Preprocessing --------------------------------

most_frequent_value <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) return(NA)

  unique_values <- unique(x)
  unique_values[which.max(tabulate(match(x, unique_values)))]
}

impute_missing_values <- function(df) {
  for (j in seq_along(df)) {
    if (is.numeric(df[[j]])) {
      med <- stats::median(df[[j]], na.rm = TRUE)
      if (!is.finite(med)) med <- 0
      df[[j]][is.na(df[[j]])] <- med
    } else {
      mode_value <- most_frequent_value(df[[j]])
      df[[j]][is.na(df[[j]])] <- mode_value
      df[[j]] <- as.factor(df[[j]])
    }
  }
  df
}

sparse_orthogonalize_matrix <- function(
    X_scaled,
    n_blocks = SPARSE_ORTHO_BINS,
    rescale_output = TRUE) {

  X_scaled <- as.matrix(X_scaled)
  out_list <- list()
  out_names <- character(0)

  for (j in seq_len(ncol(X_scaled))) {
    x <- X_scaled[, j]
    probs <- seq(0, 1, length.out = n_blocks + 1L)

    breaks <- unique(as.numeric(
      stats::quantile(
        x,
        probs = probs,
        na.rm = TRUE,
        type = 8
      )
    ))

    if (length(breaks) < 2L) next

    bin_id <- cut(
      x,
      breaks = breaks,
      include.lowest = TRUE,
      labels = FALSE
    )

    if (all(is.na(bin_id))) next

    n_retained_bins <- max(bin_id, na.rm = TRUE)

    for (b in seq_len(n_retained_bins)) {
      component <- numeric(length(x))
      idx <- which(bin_id == b)

      if (length(idx) == 0L) next

      component[idx] <- x[idx]
      out_list[[length(out_list) + 1L]] <- component
      out_names <- c(
        out_names,
        paste0(colnames(X_scaled)[j], "_b", b)
      )
    }
  }

  if (length(out_list) == 0L) {
    stop("Sparsification produced no retained component columns.")
  }

  X_sparse_raw <- do.call(cbind, out_list)
  colnames(X_sparse_raw) <- out_names

  keep_cols <- apply(
    X_sparse_raw,
    2,
    function(z) {
      s <- stats::sd(z, na.rm = TRUE)
      is.finite(s) && s > 0
    }
  )

  X_sparse_raw <- X_sparse_raw[, keep_cols, drop = FALSE]

  if (ncol(X_sparse_raw) == 0L) {
    stop("All sparsified component columns had zero variance.")
  }

  if (rescale_output) {
    X_sparse <- scale(X_sparse_raw)
    X_sparse[!is.finite(X_sparse)] <- 0
  } else {
    X_sparse <- X_sparse_raw
  }

  attr(X_sparse, "raw_sparsity") <- mean(
    abs(X_sparse_raw) < .Machine$double.eps
  )

  X_sparse
}

matrix_sparsity_ratio <- function(X) {
  mean(abs(X) < .Machine$double.eps)
}

prepare_dataset_for_clustering <- function(
    df,
    dataset_key,
    label_candidates = NULL,
    drop_candidates = NULL,
    prefer_last_column = TRUE) {

  names(df) <- sanitize_names(names(df))

  if (is.null(label_candidates)) {
    label_candidates <- c("class", "label", "category", "target", "channel")
  }
  if (is.null(drop_candidates)) {
    drop_candidates <- character(0)
  }

  last_col <- names(df)[ncol(df)]

  if (prefer_last_column) {
    label_col <- last_col
  } else {
    matches <- label_candidates[label_candidates %in% names(df)]
    label_col <- if (length(matches) > 0L) matches[1] else last_col
  }

  candidate_drop <- unique(c(
    drop_candidates,
    "id",
    "case",
    "case_number",
    "sample_id",
    "unnamed_0"
  ))
  drop_cols <- candidate_drop[candidate_drop %in% names(df)]

  labels <- as.factor(df[[label_col]])
  X_df <- df[
    ,
    setdiff(names(df), c(label_col, drop_cols)),
    drop = FALSE
  ]

  X_df <- impute_missing_values(X_df)

  model_matrix <- model.matrix(~ . - 1, data = X_df)
  model_matrix <- as.matrix(model_matrix)

  keep_cols <- apply(
    model_matrix,
    2,
    function(z) {
      s <- stats::sd(z, na.rm = TRUE)
      is.finite(s) && s > 0
    }
  )
  model_matrix <- model_matrix[, keep_cols, drop = FALSE]

  X_original <- scale(model_matrix)
  X_original[!is.finite(X_original)] <- 0

  X_sparsified <- sparse_orthogonalize_matrix(
    X_original,
    n_blocks = SPARSE_ORTHO_BINS
  )

  list(
    labels = labels,
    X_original = X_original,
    X_sparsified = X_sparsified,
    n = nrow(model_matrix),
    p = ncol(model_matrix),
    p_sparse = ncol(X_sparsified),
    true_k = nlevels(labels),
    label_col = label_col,
    label_position = ncol(df),
    drop_cols = paste(drop_cols, collapse = ";"),
    original_sparsity = matrix_sparsity_ratio(X_original),
    sparsified_raw_sparsity = attr(X_sparsified, "raw_sparsity")
  )
}

# ------------------------------ Metric functions -----------------------------

normalized_mutual_information <- function(labels_pred, labels_true) {
  tab <- table(as.factor(labels_pred), as.factor(labels_true))
  nij <- tab / sum(tab)
  pi <- rowSums(nij)
  pj <- colSums(nij)

  mutual_information <- 0
  for (i in seq_len(nrow(nij))) {
    for (j in seq_len(ncol(nij))) {
      if (nij[i, j] > 0 && pi[i] > 0 && pj[j] > 0) {
        mutual_information <- mutual_information +
          nij[i, j] * log(nij[i, j] / (pi[i] * pj[j]))
      }
    }
  }

  entropy_i <- -sum(pi[pi > 0] * log(pi[pi > 0]))
  entropy_j <- -sum(pj[pj > 0] * log(pj[pj > 0]))

  if (entropy_i <= 0 || entropy_j <= 0) return(NA_real_)

  mutual_information / sqrt(entropy_i * entropy_j)
}

clustering_purity <- function(labels_pred, labels_true) {
  if (any(is.na(labels_true))) {
    stop("True class labels contain NA values.")
  }
  if (any(is.na(labels_pred))) return(NA_real_)
  if (length(labels_pred) != length(labels_true)) return(NA_real_)

  tab <- table(as.factor(labels_pred), as.factor(labels_true))
  if (length(tab) == 0 || nrow(tab) == 0 || ncol(tab) == 0) {
    return(NA_real_)
  }

  sum(apply(tab, 1, max)) / sum(tab)
}

clustering_accuracy <- function(labels_pred, labels_true) {
  if (any(is.na(labels_true))) {
    stop("True class labels contain NA values.")
  }
  if (any(is.na(labels_pred))) return(NA_real_)
  if (length(labels_pred) != length(labels_true)) return(NA_real_)

  tab <- table(as.factor(labels_pred), as.factor(labels_true))
  if (length(tab) == 0 || nrow(tab) == 0 || ncol(tab) == 0) {
    return(NA_real_)
  }

  n_rows <- nrow(tab)
  n_cols <- ncol(tab)
  n_max <- max(n_rows, n_cols)

  score_matrix <- matrix(0, nrow = n_max, ncol = n_max)
  score_matrix[seq_len(n_rows), seq_len(n_cols)] <- tab

  assignment <- clue::solve_LSAP(score_matrix, maximum = TRUE)
  matched <- sum(
    score_matrix[cbind(seq_len(n_max), assignment)]
  )

  matched / sum(tab)
}

davies_bouldin_index <- function(X, labels) {
  labels <- as.factor(labels)
  k <- nlevels(labels)
  if (k < 2L) return(NA_real_)

  centers <- sapply(levels(labels), function(cluster_id) {
    colMeans(X[labels == cluster_id, , drop = FALSE])
  })
  if (is.vector(centers)) {
    centers <- matrix(centers, ncol = k)
  }

  within_scatter <- sapply(levels(labels), function(cluster_id) {
    Xi <- X[labels == cluster_id, , drop = FALSE]
    center_i <- colMeans(Xi)
    mean(sqrt(rowSums((sweep(Xi, 2, center_i))^2)))
  })

  center_distance <- as.matrix(dist(t(centers)))
  ratio_matrix <- matrix(NA_real_, nrow = k, ncol = k)

  for (i in seq_len(k)) {
    for (j in seq_len(k)) {
      if (i != j && center_distance[i, j] > 0) {
        ratio_matrix[i, j] <- (
          within_scatter[i] + within_scatter[j]
        ) / center_distance[i, j]
      }
    }
  }

  mean(
    apply(ratio_matrix, 1, max, na.rm = TRUE),
    na.rm = TRUE
  )
}

calinski_harabasz_index <- function(X, labels) {
  labels <- as.factor(labels)
  n <- nrow(X)
  k <- nlevels(labels)

  if (k < 2L || k >= n) return(NA_real_)

  overall_mean <- colMeans(X)
  between_sum <- 0
  within_sum <- 0

  for (cluster_id in levels(labels)) {
    Xi <- X[labels == cluster_id, , drop = FALSE]
    n_i <- nrow(Xi)
    if (n_i <= 0L) next

    center_i <- colMeans(Xi)
    between_sum <- between_sum +
      n_i * sum((center_i - overall_mean)^2)
    within_sum <- within_sum +
      sum(rowSums((sweep(Xi, 2, center_i))^2))
  }

  if (within_sum == 0) return(NA_real_)

  (between_sum / (k - 1)) / (within_sum / (n - k))
}

make_silhouette_indices <- function(n, sample_n, seed) {
  if (n <= sample_n) return(seq_len(n))

  set.seed(seed)
  sort(sample(seq_len(n), sample_n, replace = FALSE))
}

compute_metrics_for_fit <- function(
    X,
    labels_pred,
    labels_true = NULL,
    silhouette_indices = NULL) {

  if (length(labels_pred) != nrow(X) || any(is.na(labels_pred))) {
    return(list(
      silhouette = NA_real_,
      davies_bouldin = NA_real_,
      calinski_harabasz = NA_real_,
      ari = NA_real_,
      nmi = NA_real_,
      accuracy = NA_real_,
      purity = NA_real_
    ))
  }

  if (is.null(silhouette_indices)) {
    silhouette_indices <- seq_len(nrow(X))
  }

  silhouette_value <- NA_real_

  if (
    length(silhouette_indices) > 1L &&
    length(unique(labels_pred[silhouette_indices])) > 1L
  ) {
    X_sub <- X[silhouette_indices, , drop = FALSE]
    labels_sub <- labels_pred[silhouette_indices]

    silhouette_value <- tryCatch(
      {
        distance_sub <- dist(X_sub)
        silhouette_object <- cluster::silhouette(
          labels_sub,
          distance_sub
        )
        silhouette_matrix <- unclass(silhouette_object)

        if (
          is.null(dim(silhouette_matrix)) ||
          ncol(silhouette_matrix) < 3L
        ) {
          NA_real_
        } else {
          mean(silhouette_matrix[, 3], na.rm = TRUE)
        }
      },
      error = function(e) NA_real_
    )
  }

  db_value <- if (length(unique(labels_pred)) > 1L) {
    tryCatch(
      davies_bouldin_index(X, labels_pred),
      error = function(e) NA_real_
    )
  } else {
    NA_real_
  }

  ch_value <- if (length(unique(labels_pred)) > 1L) {
    tryCatch(
      calinski_harabasz_index(X, labels_pred),
      error = function(e) NA_real_
    )
  } else {
    NA_real_
  }

  ari_value <- NA_real_
  nmi_value <- NA_real_
  accuracy_value <- NA_real_
  purity_value <- NA_real_

  if (!is.null(labels_true)) {
    if (any(is.na(labels_true))) {
      stop("True class labels contain NA values.")
    }

    ari_value <- tryCatch(
      mclust::adjustedRandIndex(labels_pred, labels_true),
      error = function(e) NA_real_
    )
    nmi_value <- tryCatch(
      normalized_mutual_information(labels_pred, labels_true),
      error = function(e) NA_real_
    )
    accuracy_value <- tryCatch(
      clustering_accuracy(labels_pred, labels_true),
      error = function(e) NA_real_
    )
    purity_value <- tryCatch(
      clustering_purity(labels_pred, labels_true),
      error = function(e) NA_real_
    )
  }

  list(
    silhouette = silhouette_value,
    davies_bouldin = db_value,
    calinski_harabasz = ch_value,
    ari = ari_value,
    nmi = nmi_value,
    accuracy = accuracy_value,
    purity = purity_value
  )
}

# ------------------------------ Reproducible seeds ----------------------------

stable_string_code <- function(x) {
  values <- utf8ToInt(enc2utf8(x))
  if (length(values) == 0L) return(0L)

  as.integer(
    sum(as.double(values) * seq_along(values)) %% 100000
  )
}

make_reproducible_seed <- function(
    dataset_name,
    k,
    run_id,
    stream = 0L) {

  raw_seed <- as.double(CLUSTER_MASTER_SEED) +
    as.double(stable_string_code(dataset_name)) * 1000 +
    as.double(k) * 100 +
    as.double(run_id) +
    as.double(stream) * 10000000

  as.integer(raw_seed %% 2147483646 + 1)
}

# ------------------------------ Repeated k-means ------------------------------

run_kmeans_once <- function(X, k, fit_seed) {
  set.seed(fit_seed)

  stats::kmeans(
    X,
    centers = k,
    iter.max = KMEANS_ITER_MAX,
    nstart = KMEANS_NSTART,
    algorithm = KMEANS_ALGORITHM
  )
}

evaluate_repeated_kmeans_pair <- function(
    X_original,
    X_sparsified,
    labels = NULL,
    dataset_name,
    k_values,
    repetitions = CLUSTER_REPETITIONS,
    keep_assignments = FALSE) {

  if (nrow(X_original) != nrow(X_sparsified)) {
    stop("Original and sparsified representations must contain the same observations.")
  }
  if (!is.null(labels) && length(labels) != nrow(X_original)) {
    stop("The reference-label vector does not match the number of observations.")
  }

  representations <- list(
    original = as.matrix(X_original),
    sparsified = as.matrix(X_sparsified)
  )

  result_rows <- list()
  assignment_list <- list()
  row_counter <- 0L

  for (k in k_values) {
    if (k < 2L || k >= nrow(X_original)) {
      warning(sprintf(
        "Skipping dataset=%s, k=%d because k is outside the valid range.",
        dataset_name,
        k
      ))
      next
    }

    for (run_id in seq_len(repetitions)) {
      fit_seed <- make_reproducible_seed(
        dataset_name,
        k,
        run_id,
        stream = 1L
      )
      silhouette_seed <- make_reproducible_seed(
        dataset_name,
        k,
        run_id,
        stream = 2L
      )

      silhouette_indices <- make_silhouette_indices(
        n = nrow(X_original),
        sample_n = SILHOUETTE_SAMPLE_N,
        seed = silhouette_seed
      )

      for (representation_name in names(representations)) {
        X <- representations[[representation_name]]

        cat(sprintf(
          paste0(
            "Running dataset=%s | representation=%s | k=%d | ",
            "run=%d/%d | seed=%d | n=%d | p=%d\n"
          ),
          dataset_name,
          representation_name,
          k,
          run_id,
          repetitions,
          fit_seed,
          nrow(X),
          ncol(X)
        ))

        timed <- system.time({
          fit <- tryCatch(
            run_kmeans_once(X, k, fit_seed),
            error = function(e) {
              stop(sprintf(
                paste0(
                  "k-means failed for dataset=%s, representation=%s, ",
                  "k=%d, run=%d: %s"
                ),
                dataset_name,
                representation_name,
                k,
                run_id,
                e$message
              ))
            }
          )
        })

        metrics <- compute_metrics_for_fit(
          X = X,
          labels_pred = fit$cluster,
          labels_true = labels,
          silhouette_indices = silhouette_indices
        )

        row_counter <- row_counter + 1L
        result_rows[[row_counter]] <- data.frame(
          dataset = dataset_name,
          representation = representation_name,
          method = "kmeans",
          k = k,
          run_id = run_id,
          fit_seed = fit_seed,
          silhouette_seed = silhouette_seed,
          n_used = nrow(X),
          p_used = ncol(X),
          nstart = KMEANS_NSTART,
          iter_max = KMEANS_ITER_MAX,
          algorithm = KMEANS_ALGORITHM,
          silhouette_sample_n = length(silhouette_indices),
          silhouette = metrics$silhouette,
          davies_bouldin = metrics$davies_bouldin,
          calinski_harabasz = metrics$calinski_harabasz,
          adjusted_rand_index = metrics$ari,
          normalized_mutual_information = metrics$nmi,
          clustering_accuracy = metrics$accuracy,
          purity = metrics$purity,
          total_withinss = fit$tot.withinss,
          iterations = fit$iter,
          ifault = if (!is.null(fit$ifault)) fit$ifault else NA_integer_,
          runtime_sec = unname(timed["elapsed"]),
          stringsAsFactors = FALSE
        )

        if (keep_assignments) {
          assignment_key <- paste(
            representation_name,
            "k",
            k,
            "run",
            run_id,
            sep = "_"
          )

          assignment_list[[assignment_key]] <- data.frame(
            obs_index = seq_len(nrow(X)),
            predicted_cluster = fit$cluster,
            stringsAsFactors = FALSE
          )
        }
      }
    }
  }

  list(
    run_metrics = do.call(rbind, result_rows),
    assignments = assignment_list
  )
}

# ------------------------------ Uncertainty summaries ------------------------

metric_columns <- c(
  "adjusted_rand_index",
  "clustering_accuracy",
  "purity",
  "silhouette",
  "normalized_mutual_information",
  "davies_bouldin",
  "calinski_harabasz",
  "total_withinss",
  "runtime_sec"
)

summarize_numeric_vector <- function(
    x,
    confidence_level = CONFIDENCE_LEVEL) {

  x <- x[is.finite(x)]
  n <- length(x)

  if (n == 0L) {
    return(c(
      n_runs = 0,
      mean = NA_real_,
      sd = NA_real_,
      se = NA_real_,
      ci_lower = NA_real_,
      ci_upper = NA_real_,
      min = NA_real_,
      max = NA_real_
    ))
  }

  mean_value <- mean(x)
  sd_value <- if (n > 1L) stats::sd(x) else NA_real_
  se_value <- if (n > 1L) sd_value / sqrt(n) else NA_real_

  if (n > 1L) {
    alpha <- 1 - confidence_level
    critical_value <- stats::qt(
      1 - alpha / 2,
      df = n - 1
    )
    margin <- critical_value * se_value
    ci_lower <- mean_value - margin
    ci_upper <- mean_value + margin
  } else {
    ci_lower <- NA_real_
    ci_upper <- NA_real_
  }

  c(
    n_runs = n,
    mean = mean_value,
    sd = sd_value,
    se = se_value,
    ci_lower = ci_lower,
    ci_upper = ci_upper,
    min = min(x),
    max = max(x)
  )
}

summarize_run_metrics <- function(
    run_df,
    metrics = metric_columns) {

  group_key <- interaction(
    run_df$dataset,
    run_df$representation,
    run_df$method,
    run_df$k,
    drop = TRUE,
    lex.order = TRUE
  )

  groups <- split(run_df, group_key)
  rows <- list()
  row_counter <- 0L

  for (group_df in groups) {
    for (metric_name in metrics) {
      if (!metric_name %in% names(group_df)) next

      stats_values <- summarize_numeric_vector(
        group_df[[metric_name]]
      )

      row_counter <- row_counter + 1L
      rows[[row_counter]] <- data.frame(
        dataset = group_df$dataset[1],
        representation = group_df$representation[1],
        method = group_df$method[1],
        k = group_df$k[1],
        metric = metric_name,
        n_runs = unname(stats_values["n_runs"]),
        mean = unname(stats_values["mean"]),
        sd = unname(stats_values["sd"]),
        se = unname(stats_values["se"]),
        ci_lower = unname(stats_values["ci_lower"]),
        ci_upper = unname(stats_values["ci_upper"]),
        min = unname(stats_values["min"]),
        max = unname(stats_values["max"]),
        confidence_level = CONFIDENCE_LEVEL,
        stringsAsFactors = FALSE
      )
    }
  }

  do.call(rbind, rows)
}

summarize_paired_differences <- function(
    run_df,
    metrics = metric_columns) {

  rows <- list()
  row_counter <- 0L

  dataset_k <- unique(
    run_df[, c("dataset", "method", "k"), drop = FALSE]
  )

  for (i in seq_len(nrow(dataset_k))) {
    dataset_name <- dataset_k$dataset[i]
    method_name <- dataset_k$method[i]
    k_value <- dataset_k$k[i]

    group_df <- run_df[
      run_df$dataset == dataset_name &
        run_df$method == method_name &
        run_df$k == k_value,
      ,
      drop = FALSE
    ]

    for (metric_name in metrics) {
      if (!metric_name %in% names(group_df)) next

      original_df <- group_df[
        group_df$representation == "original",
        c("run_id", metric_name),
        drop = FALSE
      ]
      sparsified_df <- group_df[
        group_df$representation == "sparsified",
        c("run_id", metric_name),
        drop = FALSE
      ]

      names(original_df)[2] <- "original"
      names(sparsified_df)[2] <- "sparsified"

      paired_df <- merge(
        original_df,
        sparsified_df,
        by = "run_id",
        all = FALSE,
        sort = TRUE
      )

      paired_difference <- (
        paired_df$sparsified - paired_df$original
      )

      stats_values <- summarize_numeric_vector(
        paired_difference
      )

      row_counter <- row_counter + 1L
      rows[[row_counter]] <- data.frame(
        dataset = dataset_name,
        method = method_name,
        k = k_value,
        metric = metric_name,
        difference_definition = "sparsified_minus_original",
        n_pairs = unname(stats_values["n_runs"]),
        mean_difference = unname(stats_values["mean"]),
        sd_difference = unname(stats_values["sd"]),
        se_difference = unname(stats_values["se"]),
        ci_lower = unname(stats_values["ci_lower"]),
        ci_upper = unname(stats_values["ci_upper"]),
        min_difference = unname(stats_values["min"]),
        max_difference = unname(stats_values["max"]),
        confidence_level = CONFIDENCE_LEVEL,
        stringsAsFactors = FALSE
      )
    }
  }

  do.call(rbind, rows)
}

# ------------------------------ Plot functions -------------------------------

plot_metric_pairs_ci <- function(
    summary_df,
    metric_name,
    main_title,
    xlab,
    dataset_order = NULL,
    xlim_fixed = NULL) {

  sub <- summary_df[
    summary_df$metric == metric_name,
    ,
    drop = FALSE
  ]

  if (is.null(dataset_order)) {
    dataset_order <- unique(sub$dataset)
  }

  original <- sub[
    sub$representation == "original",
    ,
    drop = FALSE
  ]
  sparsified <- sub[
    sub$representation == "sparsified",
    ,
    drop = FALSE
  ]

  original <- original[
    match(dataset_order, original$dataset),
    ,
    drop = FALSE
  ]
  sparsified <- sparsified[
    match(dataset_order, sparsified$dataset),
    ,
    drop = FALSE
  ]

  valid <- !is.na(original$dataset) & !is.na(sparsified$dataset)
  original <- original[valid, , drop = FALSE]
  sparsified <- sparsified[valid, , drop = FALSE]

  dataset_labels <- gsub("_", " ", original$dataset)
  y_pos <- seq_len(nrow(original))

  if (is.null(xlim_fixed)) {
    x_range <- range(
      c(
        original$ci_lower,
        original$ci_upper,
        sparsified$ci_lower,
        sparsified$ci_upper
      ),
      na.rm = TRUE
    )

    if (!all(is.finite(x_range))) {
      x_range <- range(
        c(original$mean, sparsified$mean),
        na.rm = TRUE
      )
    }

    if (diff(x_range) == 0) {
      x_range <- x_range + c(-0.05, 0.05)
    } else {
      padding <- 0.05 * diff(x_range)
      x_range <- x_range + c(-padding, padding)
    }
  } else {
    x_range <- xlim_fixed
  }

  op <- par(mar = c(5, 20, 4, 2), xpd = NA)
  on.exit(par(op), add = TRUE)

  plot(
    original$mean,
    y_pos,
    pch = 16,
    col = "black",
    yaxt = "n",
    xlab = xlab,
    ylab = "",
    main = main_title,
    xlim = x_range
  )

  axis(
    2,
    at = y_pos,
    labels = dataset_labels,
    las = 2
  )

  segments(
    original$ci_lower,
    y_pos,
    original$ci_upper,
    y_pos,
    col = "black"
  )
  arrows(
    original$ci_lower,
    y_pos,
    original$ci_upper,
    y_pos,
    angle = 90,
    code = 3,
    length = 0.03,
    col = "black"
  )

  points(
    sparsified$mean,
    y_pos,
    pch = 17,
    col = "red"
  )
  segments(
    sparsified$ci_lower,
    y_pos,
    sparsified$ci_upper,
    y_pos,
    col = "red"
  )
  arrows(
    sparsified$ci_lower,
    y_pos,
    sparsified$ci_upper,
    y_pos,
    angle = 90,
    code = 3,
    length = 0.03,
    col = "red"
  )

  for (i in seq_len(nrow(original))) {
    segments(
      original$mean[i],
      y_pos[i],
      sparsified$mean[i],
      y_pos[i],
      col = "gray60",
      lty = 2
    )
  }

  legend(
    "topright",
    inset = c(0, -0.12),
    legend = c(
      "Original: mean and 95% CI",
      "Sparsified: mean and 95% CI"
    ),
    pch = c(16, 17),
    col = c("black", "red"),
    horiz = TRUE,
    bty = "n",
    cex = 0.9
  )
}

plot_housing_silhouette_ci <- function(summary_df) {
  sub <- summary_df[
    summary_df$metric == "silhouette",
    ,
    drop = FALSE
  ]

  original <- sub[
    sub$representation == "original",
    ,
    drop = FALSE
  ]
  sparsified <- sub[
    sub$representation == "sparsified",
    ,
    drop = FALSE
  ]

  original <- original[order(original$k), , drop = FALSE]
  sparsified <- sparsified[order(sparsified$k), , drop = FALSE]

  y_range <- range(
    c(
      original$ci_lower,
      original$ci_upper,
      sparsified$ci_lower,
      sparsified$ci_upper
    ),
    na.rm = TRUE
  )

  op <- par(mar = c(5, 5, 3, 2))
  on.exit(par(op), add = TRUE)

  plot(
    original$k - 0.04,
    original$mean,
    type = "b",
    pch = 16,
    col = "black",
    ylim = y_range,
    xaxt = "n",
    xlab = "Number of clusters (k)",
    ylab = "Average silhouette width",
    main = "Chinese housing data: repeated k-means"
  )
  axis(1, at = sort(unique(c(original$k, sparsified$k))))

  arrows(
    original$k - 0.04,
    original$ci_lower,
    original$k - 0.04,
    original$ci_upper,
    angle = 90,
    code = 3,
    length = 0.04,
    col = "black"
  )

  lines(
    sparsified$k + 0.04,
    sparsified$mean,
    type = "b",
    pch = 17,
    col = "red"
  )
  arrows(
    sparsified$k + 0.04,
    sparsified$ci_lower,
    sparsified$k + 0.04,
    sparsified$ci_upper,
    angle = 90,
    code = 3,
    length = 0.04,
    col = "red"
  )

  legend(
    "bottomright",
    legend = c(
      "Original: mean and 95% CI",
      "Sparsified: mean and 95% CI"
    ),
    pch = c(16, 17),
    col = c("black", "red"),
    bty = "n"
  )
}

plot_housing_pca_clusters <- function(
    X_original,
    original_clusters,
    X_sparsified,
    sparsified_clusters,
    years) {

  op <- par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
  on.exit(par(op), add = TRUE)

  pca_original <- prcomp(X_original)
  scores_original <- pca_original$x[, 1:2, drop = FALSE]

  plot(
    scores_original[, 1],
    scores_original[, 2],
    col = original_clusters,
    pch = 16,
    xlab = "PC1",
    ylab = "PC2",
    main = "Original data"
  )
  text(
    scores_original[, 1],
    scores_original[, 2],
    labels = years,
    pos = rep(c(1, 3, 2, 4), length.out = length(years)),
    cex = 0.68
  )

  pca_sparsified <- prcomp(X_sparsified)
  scores_sparsified <- pca_sparsified$x[, 1:2, drop = FALSE]

  plot(
    scores_sparsified[, 1],
    scores_sparsified[, 2],
    col = sparsified_clusters,
    pch = 16,
    xlab = "PC1",
    ylab = "PC2",
    main = "Sparsified data"
  )
  text(
    scores_sparsified[, 1],
    scores_sparsified[, 2],
    labels = years,
    pos = rep(c(1, 3, 2, 4), length.out = length(years)),
    cex = 0.68
  )
}

clustering_protocol


Warning message:
"package 'readxl' was built under R version 4.4.3"
Warning message:
"package 'mclust' was built under R version 4.4.3"
Package 'mclust' version 6.1.2
Type 'citation("mclust")' for citing this R package in publications.

Warning message:
"package 'clue' was built under R version 4.4.3"


item,value
<chr>,<chr>
Independent repetitions,30
Random starts per repetition,30
k-means algorithm,Hartigan-Wong
Maximum iterations,100
Master seed,20260716
Silhouette sample limit,1000
Confidence level,0.95
Sparsification intervals,4


## 11. Load and preprocess the UCI datasets

The notebook searches for the candidate UCI data files listed below. Files that are not present are skipped. The paper reports the ten datasets that are successfully retained for analysis.

For every retained dataset:

1. the final prepared column is treated as the reference-label column;
2. the label is excluded from clustering and used only for external evaluation;
3. numerical missing values are replaced by the feature median;
4. nonnumerical missing values are replaced by the most frequent category;
5. categorical predictors are expanded using `model.matrix(~ . - 1)`;
6. zero-variance columns are removed;
7. the retained original predictors are standardized;
8. each standardized predictor is divided into four type-8 empirical-quantile intervals;
9. the retained sparse components are standardized before clustering.


In [12]:
# Load and preprocess the candidate UCI datasets ------------------------------

uci_specs <- list(
  breast_tissue = list(
    aliases = c("breast_tissue"),
    label_candidates = c("class", "label"),
    drop_candidates = c("case", "case_number")
  ),
  hcv = list(
    aliases = c("hcv"),
    label_candidates = c("class", "category", "label"),
    drop_candidates = c("unnamed_0")
  ),
  wholesale_customers = list(
    aliases = c("wholesale_customers"),
    label_candidates = c("class", "label", "channel"),
    drop_candidates = c("region", "customer_id")
  ),
  pen_recognition_handwritten_digits = list(
    aliases = c("pen_recognition_handwritten_digits"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  libras_movement = list(
    aliases = c("libras_movement"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  vehicle_silhouettes = list(
    aliases = c("vehicle_silhouettes", "vehicle_silhouette"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  statlog_image_segmentation = list(
    aliases = c(
      "statlog_image_segmentation",
      "image_segmentation",
      "statlog_segment"
    ),
    label_candidates = c("class", "label"),
    drop_candidates = c("region_pixel_count")
  ),
  vertebral_column_3classes = list(
    aliases = c(
      "vertebral_column_3classes",
      "vertebral_column_3_classes",
      "vertebral_3classes"
    ),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  vertebral_column_2classes = list(
    aliases = c(
      "vertebral_column_2classes",
      "vertebral_column_2_classes",
      "vertebral_2classes"
    ),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  user_knowledge_modeling = list(
    aliases = c(
      "user_knowledge_modeling",
      "user_knowledge_modelling"
    ),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  banknote_authentication = list(
    aliases = c(
      "banknote_authentication",
      "banknote_authentication_dataset",
      "banknote_auth"
    ),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  wine = list(
    aliases = c("wine"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  )
)

uci_prepared <- list()
uci_overview <- data.frame()

for (dataset_name in names(uci_specs)) {
  spec <- uci_specs[[dataset_name]]
  path <- find_data_file_multi(spec$aliases)

  if (is.na(path)) {
    warning(sprintf(
      "Dataset file not found for %s; this dataset will be skipped.",
      dataset_name
    ))
    next
  }

  raw_df <- read_tabular_file(path)

  prepared <- prepare_dataset_for_clustering(
    df = raw_df,
    dataset_key = dataset_name,
    label_candidates = spec$label_candidates,
    drop_candidates = spec$drop_candidates,
    prefer_last_column = TRUE
  )

  if (any(is.na(prepared$labels))) {
    stop(sprintf(
      "Dataset %s has NA values in the reference-label column.",
      dataset_name
    ))
  }

  cat(sprintf(
    paste0(
      "Prepared dataset=%s | n=%d | p=%d | p_s=%d | ",
      "reference classes=%d\n"
    ),
    dataset_name,
    prepared$n,
    prepared$p,
    prepared$p_sparse,
    prepared$true_k
  ))

  uci_prepared[[dataset_name]] <- prepared

  uci_overview <- rbind(
    uci_overview,
    data.frame(
      dataset = dataset_name,
      file_path = path,
      n = prepared$n,
      p_after_encoding = prepared$p,
      p_sparse_components = prepared$p_sparse,
      label_col = prepared$label_col,
      label_position = prepared$label_position,
      dropped_cols = prepared$drop_cols,
      true_k = prepared$true_k,
      original_processed_zero_ratio = round(
        prepared$original_sparsity,
        6
      ),
      sparsified_raw_zero_ratio = round(
        prepared$sparsified_raw_sparsity,
        6
      ),
      stringsAsFactors = FALSE
    )
  )
}

if (length(uci_prepared) == 0L) {
  stop("No UCI datasets were loaded. Place the data files in data/ or beside the notebook.")
}

write.csv(
  uci_overview,
  "outputs/uci_dataset_overview.csv",
  row.names = FALSE
)

cat(
  "Datasets loaded:",
  paste(names(uci_prepared), collapse = ", "),
  "\n"
)
cat(
  "The last prepared column is used as the reference-label column.\n"
)
cat(sprintf(
  "Quantile intervals per encoded feature: %d\n",
  SPARSE_ORTHO_BINS
))

uci_overview


Prepared dataset=breast_tissue | n=106 | p=9 | p_s=36 | reference classes=6
Prepared dataset=hcv | n=615 | p=13 | p_s=46 | reference classes=5
Prepared dataset=wholesale_customers | n=440 | p=7 | p_s=25 | reference classes=3
Prepared dataset=pen_recognition_handwritten_digits | n=10992 | p=16 | p_s=59 | reference classes=10
Prepared dataset=libras_movement | n=360 | p=90 | p_s=360 | reference classes=15
Prepared dataset=vehicle_silhouettes | n=846 | p=18 | p_s=72 | reference classes=4
Prepared dataset=statlog_image_segmentation | n=2310 | p=18 | p_s=66 | reference classes=7
Prepared dataset=vertebral_column_3classes | n=310 | p=6 | p_s=24 | reference classes=3
Prepared dataset=vertebral_column_2classes | n=310 | p=6 | p_s=24 | reference classes=2
Prepared dataset=user_knowledge_modeling | n=403 | p=5 | p_s=20 | reference classes=5
Prepared dataset=banknote_authentication | n=1372 | p=4 | p_s=16 | reference classes=2
Prepared dataset=wine | n=178 | p=13 | p_s=52 | reference classes=3
Da

dataset,file_path,n,p_after_encoding,p_sparse_components,label_col,label_position,dropped_cols,true_k,original_processed_zero_ratio,sparsified_raw_zero_ratio
<chr>,<chr>,<int>,<int>,<int>,<chr>,<int>,<chr>,<int>,<dbl>,<dbl>
breast_tissue,data/breast_tissue.csv,106,9,36,class,11,case,6,0,0.750000
hcv,data/hcv.csv,615,13,46,class,14,id,5,0,0.717391
wholesale_customers,data/wholesale_customers.csv,440,7,25,class,8,,3,0,0.720000
pen_recognition_handwritten_digits,data/pen_recognition_handwritten_digits.csv,10992,16,59,class,17,,10,0,0.728814
libras_movement,data/libras_movement.csv,360,90,360,class,91,,15,0,0.750000
vehicle_silhouettes,data/vehicle_silhouettes.csv,846,18,72,class,19,,4,0,0.750000
statlog_image_segmentation,data/statlog_image_segmentation.csv,2310,18,66,class,20,region_pixel_count,7,0,0.727273
vertebral_column_3classes,data/vertebral_column_3classes.csv,310,6,24,class,7,,3,0,0.750000
vertebral_column_2classes,data/vertebral_column_2classes.csv,310,6,24,class,7,,2,0,0.750000


In [13]:
# Dataset diagnostics ----------------------------------------------------------

uci_diagnostics <- data.frame()

for (dataset_name in names(uci_prepared)) {
  prepared <- uci_prepared[[dataset_name]]
  class_sizes <- table(prepared$labels)

  uci_diagnostics <- rbind(
    uci_diagnostics,
    data.frame(
      dataset = dataset_name,
      n = prepared$n,
      p = prepared$p,
      p_sparse = prepared$p_sparse,
      true_k = prepared$true_k,
      label_na = sum(is.na(prepared$labels)),
      original_na = sum(is.na(prepared$X_original)),
      sparsified_na = sum(is.na(prepared$X_sparsified)),
      min_class_size = min(class_sizes),
      max_class_size = max(class_sizes),
      original_unique_rows = nrow(
        unique(as.data.frame(prepared$X_original))
      ),
      sparsified_unique_rows = nrow(
        unique(as.data.frame(prepared$X_sparsified))
      ),
      stringsAsFactors = FALSE
    )
  )
}

write.csv(
  uci_diagnostics,
  "outputs/uci_dataset_diagnostics.csv",
  row.names = FALSE
)

uci_diagnostics


dataset,n,p,p_sparse,true_k,label_na,original_na,sparsified_na,min_class_size,max_class_size,original_unique_rows,sparsified_unique_rows
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
breast_tissue,106,9,36,6,0,0,0,14,22,105,105
hcv,615,13,46,5,0,0,0,7,533,615,615
wholesale_customers,440,7,25,3,0,0,0,47,316,440,440
pen_recognition_handwritten_digits,10992,16,59,10,0,0,0,1055,1144,10992,10992
libras_movement,360,90,360,15,0,0,0,24,24,330,330
vehicle_silhouettes,846,18,72,4,0,0,0,199,218,846,846
statlog_image_segmentation,2310,18,66,7,0,0,0,330,330,2086,2086
vertebral_column_3classes,310,6,24,3,0,0,0,60,150,310,310
vertebral_column_2classes,310,6,24,2,0,0,0,100,210,310,310


In [14]:
# Reproducibility checks -------------------------------------------------------

stopifnot(CLUSTER_REPETITIONS >= 2L)
stopifnot(KMEANS_NSTART >= 1L)
stopifnot(KMEANS_ITER_MAX >= 1L)
stopifnot(KMEANS_ALGORITHM %in% c(
  "Hartigan-Wong",
  "Lloyd",
  "Forgy",
  "MacQueen"
))

seed_check <- data.frame(
  run_id = seq_len(min(5L, CLUSTER_REPETITIONS)),
  fit_seed = vapply(
    seq_len(min(5L, CLUSTER_REPETITIONS)),
    function(run_id) {
      make_reproducible_seed(
        dataset_name = names(uci_prepared)[1],
        k = uci_prepared[[1]]$true_k,
        run_id = run_id,
        stream = 1L
      )
    },
    integer(1)
  ),
  silhouette_seed = vapply(
    seq_len(min(5L, CLUSTER_REPETITIONS)),
    function(run_id) {
      make_reproducible_seed(
        dataset_name = names(uci_prepared)[1],
        k = uci_prepared[[1]]$true_k,
        run_id = run_id,
        stream = 2L
      )
    },
    integer(1)
  )
)

cat(
  "The same fit seed and silhouette seed will be used for the original\n",
  "and sparsified representations within every paired repetition.\n"
)

seed_check


The same fit seed and silhouette seed will be used for the original
 and sparsified representations within every paired repetition.


run_id,fit_seed,silhouette_seed
<int>,<int>,<int>
1,40219318,50219318
2,40219319,50219319
3,40219320,50219320
4,40219321,50219321
5,40219322,50219322


## 12. Repeated \(k\)-means evaluation on the UCI datasets

For each retained dataset, \(k\) is fixed to the number of reference classes. The complete experiment is repeated 30 times. Each repetition uses a recorded seed and 30 internal random starts. The original and sparsified representations use the same seed sequence.

The notebook saves:

- one row per dataset, representation, and repetition;
- mean, standard deviation, standard error, and 95% confidence interval for each metric;
- paired differences \(M_{\text{sparsified}}-M_{\text{original}}\) and their 95% confidence intervals;
- the full seed plan needed to reproduce every run.


In [15]:
# Repeated UCI k-means experiment ---------------------------------------------

uci_run_metrics <- data.frame()

for (dataset_name in names(uci_prepared)) {
  prepared <- uci_prepared[[dataset_name]]

  evaluation <- evaluate_repeated_kmeans_pair(
    X_original = prepared$X_original,
    X_sparsified = prepared$X_sparsified,
    labels = prepared$labels,
    dataset_name = dataset_name,
    k_values = prepared$true_k,
    repetitions = CLUSTER_REPETITIONS,
    keep_assignments = FALSE
  )

  uci_run_metrics <- rbind(
    uci_run_metrics,
    evaluation$run_metrics
  )
}

uci_metric_summary <- summarize_run_metrics(
  uci_run_metrics
)

uci_paired_difference_summary <- summarize_paired_differences(
  uci_run_metrics
)

uci_seed_plan <- unique(
  uci_run_metrics[
    ,
    c(
      "dataset",
      "k",
      "run_id",
      "fit_seed",
      "silhouette_seed",
      "silhouette_sample_n"
    ),
    drop = FALSE
  ]
)
uci_seed_plan <- uci_seed_plan[
  order(
    uci_seed_plan$dataset,
    uci_seed_plan$k,
    uci_seed_plan$run_id
  ),
  ,
  drop = FALSE
]

# Main metrics requested by Reviewer 2, Comment 5.
main_metric_names <- c(
  "adjusted_rand_index",
  "clustering_accuracy",
  "purity",
  "silhouette"
)

uci_main_metric_summary <- uci_metric_summary[
  uci_metric_summary$metric %in% main_metric_names,
  ,
  drop = FALSE
]

uci_main_paired_summary <- uci_paired_difference_summary[
  uci_paired_difference_summary$metric %in% main_metric_names,
  ,
  drop = FALSE
]

write.csv(
  uci_run_metrics,
  "outputs/uci_clustering_true_k_metrics_by_run.csv",
  row.names = FALSE
)
write.csv(
  uci_metric_summary,
  "outputs/uci_clustering_true_k_metric_summary.csv",
  row.names = FALSE
)
write.csv(
  uci_paired_difference_summary,
  "outputs/uci_clustering_true_k_paired_difference_summary.csv",
  row.names = FALSE
)
write.csv(
  uci_seed_plan,
  "outputs/uci_clustering_true_k_seed_plan.csv",
  row.names = FALSE
)
write.csv(
  uci_main_metric_summary,
  "outputs/uci_clustering_true_k_main_metrics_summary.csv",
  row.names = FALSE
)
write.csv(
  uci_main_paired_summary,
  "outputs/uci_clustering_true_k_main_paired_differences.csv",
  row.names = FALSE
)

capture.output(
  list(
    protocol = clustering_protocol,
    dataset_overview = uci_overview,
    main_metric_summary = uci_main_metric_summary,
    main_paired_difference_summary = uci_main_paired_summary,
    seed_plan = uci_seed_plan
  ),
  file = "outputs/uci_clustering_repeated_run_summary.txt"
)

uci_main_metric_summary
uci_main_paired_summary


Running dataset=breast_tissue | representation=original | k=6 | run=1/30 | seed=40219318 | n=106 | p=9
Running dataset=breast_tissue | representation=sparsified | k=6 | run=1/30 | seed=40219318 | n=106 | p=36
Running dataset=breast_tissue | representation=original | k=6 | run=2/30 | seed=40219319 | n=106 | p=9
Running dataset=breast_tissue | representation=sparsified | k=6 | run=2/30 | seed=40219319 | n=106 | p=36
Running dataset=breast_tissue | representation=original | k=6 | run=3/30 | seed=40219320 | n=106 | p=9
Running dataset=breast_tissue | representation=sparsified | k=6 | run=3/30 | seed=40219320 | n=106 | p=36
Running dataset=breast_tissue | representation=original | k=6 | run=4/30 | seed=40219321 | n=106 | p=9
Running dataset=breast_tissue | representation=sparsified | k=6 | run=4/30 | seed=40219321 | n=106 | p=36
Running dataset=breast_tissue | representation=original | k=6 | run=5/30 | seed=40219322 | n=106 | p=9
Running dataset=breast_tissue | representation=sparsified | k

,dataset,representation,method,k,metric,n_runs,mean,sd,se,ci_lower,ci_upper,min,max,confidence_level
,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,banknote_authentication,original,kmeans,2,adjusted_rand_index,30,0.01321583,0.000000000,0.0000000000,0.01321583,0.01321583,0.01321583,0.01321583,0.95
2,banknote_authentication,original,kmeans,2,clustering_accuracy,30,0.55903790,0.000000000,0.0000000000,0.55903790,0.55903790,0.55903790,0.55903790,0.95
3,banknote_authentication,original,kmeans,2,purity,30,0.55903790,0.000000000,0.0000000000,0.55903790,0.55903790,0.55903790,0.55903790,0.95
4,banknote_authentication,original,kmeans,2,silhouette,30,0.32856519,0.003466490,0.0006328916,0.32727079,0.32985960,0.32102530,0.33697496,0.95
10,banknote_authentication,sparsified,kmeans,2,adjusted_rand_index,30,0.02362336,0.000000000,0.0000000000,0.02362336,0.02362336,0.02362336,0.02362336,0.95
11,banknote_authentication,sparsified,kmeans,2,clustering_accuracy,30,0.57798834,0.000000000,0.0000000000,0.57798834,0.57798834,0.57798834,0.57798834,0.95
12,banknote_authentication,sparsified,kmeans,2,purity,30,0.57798834,0.000000000,0.0000000000,0.57798834,0.57798834,0.57798834,0.57798834,0.95
13,banknote_authentication,sparsified,kmeans,2,silhouette,30,0.13947653,0.001504202,0.0002746284,0.13891485,0.14003820,0.13640647,0.14306293,0.95
19,breast_tissue,original,kmeans,6,adjusted_rand_index,30,0.29100666,0.009812835,0.0017915704,0.28734249,0.29467084,0.28508012,0.30940007,0.95


,dataset,method,k,metric,difference_definition,n_pairs,mean_difference,sd_difference,se_difference,ci_lower,ci_upper,min_difference,max_difference,confidence_level
,<chr>,<chr>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,breast_tissue,kmeans,6,adjusted_rand_index,sparsified_minus_original,30,0.0698459821,0.009812835,0.0017915704,0.0661818093,0.0735101550,0.0514525746,0.0757725207,0.95
2,breast_tissue,kmeans,6,clustering_accuracy,sparsified_minus_original,30,0.0396226415,0.012729522,0.0023240821,0.0348693599,0.0443759232,0.0188679245,0.0471698113,0.95
3,breast_tissue,kmeans,6,purity,sparsified_minus_original,30,0.0251572327,0.017223980,0.0031446541,0.0187256929,0.0315887725,0.0000000000,0.0471698113,0.95
4,breast_tissue,kmeans,6,silhouette,sparsified_minus_original,30,-0.1819085601,0.036543755,0.0066719463,-0.1955542224,-0.1682628978,-0.2132489594,-0.1170367704,0.95
10,hcv,kmeans,5,adjusted_rand_index,sparsified_minus_original,30,-0.0197059847,0.036607413,0.0066835686,-0.0333754174,-0.0060365520,-0.0954589071,0.0269992586,0.95
11,hcv,kmeans,5,clustering_accuracy,sparsified_minus_original,30,-0.0788617886,0.074637817,0.0136269387,-0.1067320075,-0.0509915697,-0.2308943089,0.0113821138,0.95
12,hcv,kmeans,5,purity,sparsified_minus_original,30,-0.0003252033,0.003899635,0.0007119726,-0.0017813507,0.0011309442,-0.0065040650,0.0081300813,0.95
13,hcv,kmeans,5,silhouette,sparsified_minus_original,30,-0.1681574532,0.033700995,0.0061529316,-0.1807416114,-0.1555732950,-0.2275849307,-0.1386571039,0.95
19,wholesale_customers,kmeans,3,adjusted_rand_index,sparsified_minus_original,30,0.0001846758,0.000000000,0.0000000000,0.0001846758,0.0001846758,0.0001846758,0.0001846758,0.95


## 13. Save UCI clustering figures with 95% confidence intervals

The figures retain the filenames used by the manuscript. Each point is the mean across the independent repetitions, and each error bar is a 95% \(t\)-confidence interval across the 30 runs.


In [16]:
# Save repeated-run UCI clustering figures ------------------------------------

uci_dataset_order <- names(uci_prepared)

save_plot_pdf(
  "figure10_uci_ari_at_true_k.pdf",
  plot_metric_pairs_ci(
    summary_df = uci_main_metric_summary,
    metric_name = "adjusted_rand_index",
    main_title = "ARI at the label-based number of clusters",
    xlab = "Adjusted Rand Index",
    dataset_order = uci_dataset_order
  ),
  width = 9,
  height = 5.5
)

save_plot_pdf(
  "figure11_uci_accuracy_at_true_k.pdf",
  plot_metric_pairs_ci(
    summary_df = uci_main_metric_summary,
    metric_name = "clustering_accuracy",
    main_title = "Accuracy at the label-based number of clusters",
    xlab = "Clustering accuracy",
    dataset_order = uci_dataset_order,
    xlim_fixed = c(0, 1)
  ),
  width = 9,
  height = 5.5
)

save_plot_pdf(
  "figure12_uci_purity_at_true_k.pdf",
  plot_metric_pairs_ci(
    summary_df = uci_main_metric_summary,
    metric_name = "purity",
    main_title = "Purity at the label-based number of clusters",
    xlab = "Purity",
    dataset_order = uci_dataset_order,
    xlim_fixed = c(0, 1)
  ),
  width = 9,
  height = 5.5
)

cat(
  "Saved UCI figures using repeated-run means and 95% confidence intervals.\n"
)


Warning message in arrows(original$ci_lower, y_pos, original$ci_upper, y_pos, angle = 90, :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(original$ci_lower, y_pos, original$ci_upper, y_pos, angle = 90, :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(original$ci_lower, y_pos, original$ci_upper, y_pos, angle = 90, :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(original$ci_lower, y_pos, original$ci_upper, y_pos, angle = 90, :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(original$ci_lower, y_pos, original$ci_upper, y_pos, angle = 90, :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(sparsified$ci_lower, y_pos, sparsified$ci_upper, y_pos, :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(sparsified$ci_lower, y_pos, sparsified$ci_upper, y_pos, :
"zero-

Saved UCI figures using repeated-run means and 95% confidence intervals.


## 14. Repeated exploratory clustering on the Chinese housing dataset

The housing analysis remains exploratory. The years are clustered using predictors `x1`--`x7`; the housing-price response `y` is excluded. The repeated-run protocol is the same as for the UCI data, except that \(k\in\{2,3,4,5\}\) is evaluated.

For each representation, the displayed PCA partition uses:

1. the value of \(k\) with the highest **mean** silhouette width across repetitions; and
2. the repetition whose silhouette width is closest to that mean, providing a reproducible representative partition rather than selecting the single most favorable run.


In [17]:
# Repeated clustering on the Chinese housing dataset --------------------------

housing_file <- find_data_file_multi(
  c("regression_data_1997_2023")
)

if (is.na(housing_file)) {
  stop(
    "regression_data_1997_2023 was not found in data/ or the current directory."
  )
}

housing_df <- read.csv(
  housing_file,
  stringsAsFactors = FALSE
)

if (!"year" %in% names(housing_df)) {
  housing_df$year <- 1997:2023
}

housing_X <- as.matrix(
  housing_df[
    ,
    c("x1", "x2", "x3", "x4", "x5", "x6", "x7"),
    drop = FALSE
  ]
)

housing_X_original <- scale(housing_X)
housing_X_original[!is.finite(housing_X_original)] <- 0

housing_X_sparsified <- sparse_orthogonalize_matrix(
  housing_X_original,
  n_blocks = SPARSE_ORTHO_BINS
)

housing_evaluation <- evaluate_repeated_kmeans_pair(
  X_original = housing_X_original,
  X_sparsified = housing_X_sparsified,
  labels = NULL,
  dataset_name = "housing_years",
  k_values = 2:5,
  repetitions = CLUSTER_REPETITIONS,
  keep_assignments = TRUE
)

housing_run_metrics <- housing_evaluation$run_metrics
housing_metric_summary <- summarize_run_metrics(
  housing_run_metrics
)
housing_paired_difference_summary <- summarize_paired_differences(
  housing_run_metrics
)

write.csv(
  housing_run_metrics,
  "outputs/housing_clustering_internal_metrics_by_run.csv",
  row.names = FALSE
)
write.csv(
  housing_metric_summary,
  "outputs/housing_clustering_internal_metric_summary.csv",
  row.names = FALSE
)
write.csv(
  housing_paired_difference_summary,
  "outputs/housing_clustering_paired_difference_summary.csv",
  row.names = FALSE
)

housing_silhouette_summary <- housing_metric_summary[
  housing_metric_summary$metric == "silhouette",
  ,
  drop = FALSE
]

select_best_k <- function(summary_df, representation_name) {
  sub <- summary_df[
    summary_df$representation == representation_name,
    ,
    drop = FALSE
  ]

  sub$k[which.max(sub$mean)]
}

select_representative_run <- function(
    run_df,
    representation_name,
    k_value,
    target_mean) {

  sub <- run_df[
    run_df$representation == representation_name &
      run_df$k == k_value,
    ,
    drop = FALSE
  ]

  valid <- is.finite(sub$silhouette)
  sub <- sub[valid, , drop = FALSE]

  if (nrow(sub) == 0L) {
    stop("No valid silhouette results were available for representative-run selection.")
  }

  sub$run_id[
    which.min(abs(sub$silhouette - target_mean))
  ]
}

best_original_k <- select_best_k(
  housing_silhouette_summary,
  "original"
)
best_sparsified_k <- select_best_k(
  housing_silhouette_summary,
  "sparsified"
)

original_mean_silhouette <- housing_silhouette_summary$mean[
  housing_silhouette_summary$representation == "original" &
    housing_silhouette_summary$k == best_original_k
][1]

sparsified_mean_silhouette <- housing_silhouette_summary$mean[
  housing_silhouette_summary$representation == "sparsified" &
    housing_silhouette_summary$k == best_sparsified_k
][1]

representative_original_run <- select_representative_run(
  housing_run_metrics,
  representation_name = "original",
  k_value = best_original_k,
  target_mean = original_mean_silhouette
)

representative_sparsified_run <- select_representative_run(
  housing_run_metrics,
  representation_name = "sparsified",
  k_value = best_sparsified_k,
  target_mean = sparsified_mean_silhouette
)

original_assignment_key <- paste(
  "original",
  "k",
  best_original_k,
  "run",
  representative_original_run,
  sep = "_"
)
sparsified_assignment_key <- paste(
  "sparsified",
  "k",
  best_sparsified_k,
  "run",
  representative_sparsified_run,
  sep = "_"
)

original_clusters <- housing_evaluation$assignments[[original_assignment_key]]$predicted_cluster
sparsified_clusters <- housing_evaluation$assignments[[sparsified_assignment_key]]$predicted_cluster

housing_assignments <- data.frame(
  year = housing_df$year,
  original_k = best_original_k,
  original_representative_run = representative_original_run,
  original_cluster = original_clusters,
  sparsified_k = best_sparsified_k,
  sparsified_representative_run = representative_sparsified_run,
  sparsified_cluster = sparsified_clusters,
  stringsAsFactors = FALSE
)

housing_selected_models <- data.frame(
  representation = c("original", "sparsified"),
  selected_k = c(best_original_k, best_sparsified_k),
  mean_silhouette = c(
    original_mean_silhouette,
    sparsified_mean_silhouette
  ),
  representative_run = c(
    representative_original_run,
    representative_sparsified_run
  ),
  selection_rule = c(
    "highest mean silhouette; run closest to mean",
    "highest mean silhouette; run closest to mean"
  ),
  stringsAsFactors = FALSE
)

write.csv(
  housing_assignments,
  "outputs/housing_clustering_representative_assignments.csv",
  row.names = FALSE
)
write.csv(
  housing_selected_models,
  "outputs/housing_clustering_selected_models.csv",
  row.names = FALSE
)

save_plot_pdf(
  "figure13_housing_internal_metrics.pdf",
  plot_housing_silhouette_ci(
    housing_metric_summary
  ),
  width = 8,
  height = 7
)

save_plot_pdf(
  "figure14_housing_pca_clusters.pdf",
  plot_housing_pca_clusters(
    X_original = housing_X_original,
    original_clusters = original_clusters,
    X_sparsified = housing_X_sparsified,
    sparsified_clusters = sparsified_clusters,
    years = housing_df$year
  ),
  width = 10,
  height = 5
)

capture.output(
  list(
    protocol = clustering_protocol,
    selected_models = housing_selected_models,
    silhouette_summary = housing_silhouette_summary,
    paired_difference_summary = housing_paired_difference_summary
  ),
  file = "outputs/housing_clustering_repeated_run_summary.txt"
)

housing_selected_models
housing_silhouette_summary
housing_assignments


Running dataset=housing_years | representation=original | k=2 | run=1/30 | seed=40092918 | n=27 | p=7
Running dataset=housing_years | representation=sparsified | k=2 | run=1/30 | seed=40092918 | n=27 | p=28
Running dataset=housing_years | representation=original | k=2 | run=2/30 | seed=40092919 | n=27 | p=7
Running dataset=housing_years | representation=sparsified | k=2 | run=2/30 | seed=40092919 | n=27 | p=28
Running dataset=housing_years | representation=original | k=2 | run=3/30 | seed=40092920 | n=27 | p=7
Running dataset=housing_years | representation=sparsified | k=2 | run=3/30 | seed=40092920 | n=27 | p=28
Running dataset=housing_years | representation=original | k=2 | run=4/30 | seed=40092921 | n=27 | p=7
Running dataset=housing_years | representation=sparsified | k=2 | run=4/30 | seed=40092921 | n=27 | p=28
Running dataset=housing_years | representation=original | k=2 | run=5/30 | seed=40092922 | n=27 | p=7
Running dataset=housing_years | representation=sparsified | k=2 | run=

Warning message in arrows(original$k - 0.04, original$ci_lower, original$k - 0.04, :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(original$k - 0.04, original$ci_lower, original$k - 0.04, :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(original$k - 0.04, original$ci_lower, original$k - 0.04, :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(sparsified$k + 0.04, sparsified$ci_lower, sparsified$k + :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(sparsified$k + 0.04, sparsified$ci_lower, sparsified$k + :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(sparsified$k + 0.04, sparsified$ci_lower, sparsified$k + :
"zero-length arrow is of indeterminate angle and so skipped"
Warning message in arrows(sparsified$k + 0.04, sparsified$ci_lower, sparsified$k + :
"zero-length arrow is of indeterminate 

representation,selected_k,mean_silhouette,representative_run,selection_rule
<chr>,<int>,<dbl>,<int>,<chr>
original,2,0.6565444,1,highest mean silhouette; run closest to mean
sparsified,4,0.6759242,1,highest mean silhouette; run closest to mean


,dataset,representation,method,k,metric,n_runs,mean,sd,se,ci_lower,ci_upper,min,max,confidence_level
,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
4,housing_years,original,kmeans,2,silhouette,30,0.6565444,0.000000000,0.0000000000,0.6565444,0.6565444,0.6565444,0.6565444,0.95
13,housing_years,original,kmeans,3,silhouette,30,0.6014447,0.000000000,0.0000000000,0.6014447,0.6014447,0.6014447,0.6014447,0.95
22,housing_years,original,kmeans,4,silhouette,30,0.5769435,0.000000000,0.0000000000,0.5769435,0.5769435,0.5769435,0.5769435,0.95
31,housing_years,original,kmeans,5,silhouette,30,0.5456305,0.003258272,0.0005948764,0.5444138,0.5468471,0.5283790,0.5462253,0.95
40,housing_years,sparsified,kmeans,2,silhouette,30,0.3969590,0.000000000,0.0000000000,0.3969590,0.3969590,0.3969590,0.3969590,0.95
49,housing_years,sparsified,kmeans,3,silhouette,30,0.5659337,0.000000000,0.0000000000,0.5659337,0.5659337,0.5659337,0.5659337,0.95
58,housing_years,sparsified,kmeans,4,silhouette,30,0.6759242,0.000000000,0.0000000000,0.6759242,0.6759242,0.6759242,0.6759242,0.95
67,housing_years,sparsified,kmeans,5,silhouette,30,0.6580134,0.000000000,0.0000000000,0.6580134,0.6580134,0.6580134,0.6580134,0.95


year,original_k,original_representative_run,original_cluster,sparsified_k,sparsified_representative_run,sparsified_cluster
<int>,<int>,<int>,<int>,<int>,<int>,<int>
1997,2,1,1,4,1,3
1998,2,1,1,4,1,3
1999,2,1,1,4,1,3
2000,2,1,1,4,1,3
2001,2,1,1,4,1,3
2002,2,1,1,4,1,3
2003,2,1,1,4,1,3
2004,2,1,1,4,1,2
2005,2,1,1,4,1,2


## 15. Files created by the revised clustering section

### Protocol and diagnostics

- `outputs/clustering_repeated_run_protocol.csv`
- `outputs/uci_dataset_overview.csv`
- `outputs/uci_dataset_diagnostics.csv`

### UCI repeated-run results

- `outputs/uci_clustering_true_k_metrics_by_run.csv`
- `outputs/uci_clustering_true_k_metric_summary.csv`
- `outputs/uci_clustering_true_k_paired_difference_summary.csv`
- `outputs/uci_clustering_true_k_seed_plan.csv`
- `outputs/uci_clustering_true_k_main_metrics_summary.csv`
- `outputs/uci_clustering_true_k_main_paired_differences.csv`
- `outputs/uci_clustering_repeated_run_summary.txt`
- `outputs/figure10_uci_ari_at_true_k.pdf`
- `outputs/figure11_uci_accuracy_at_true_k.pdf`
- `outputs/figure12_uci_purity_at_true_k.pdf`

### Chinese housing repeated-run results

- `outputs/housing_clustering_internal_metrics_by_run.csv`
- `outputs/housing_clustering_internal_metric_summary.csv`
- `outputs/housing_clustering_paired_difference_summary.csv`
- `outputs/housing_clustering_representative_assignments.csv`
- `outputs/housing_clustering_selected_models.csv`
- `outputs/housing_clustering_repeated_run_summary.txt`
- `outputs/figure13_housing_internal_metrics.pdf`
- `outputs/figure14_housing_pca_clusters.pdf`

The run-level files record every seed, metric, \(k\)-means setting, and runtime. The summary files provide means, standard deviations, and 95% confidence intervals. The paired-difference files summarize sparsified-minus-original differences using matched repetition IDs.
